In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
#from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI # this is one of the LLM abstraction that will respond to the invoke()
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import streamlit as st
import gradio as gr



In [ ]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# How many characters in all the documents?

knowledge_base_path = "knowledge_base_markdown/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(files)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count= len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
folders = glob.glob("knowledge_base_markdown/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    print("Doc Type = " + doc_type)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})

    folder_docs = loader.load()
    print(folder_docs)
    for doc in folder_docs:
        md_path = doc.metadata["source"]

        filename = os.path.splitext(os.path.basename(md_path))[0]

        raw_extension_map = {
            "pdf": ".pdf",
            "word": ".docx",
            "excel": ".xlsx",
            "images": ".png",
            "ppt": ".pptx"
        }

        raw_path = os.path.join(
            "knowledge_base_raw",
            doc_type,
            filename + raw_extension_map.get(doc_type, "")
        ).replace("\\", "/")


        doc.metadata["doc_type"] = doc_type
        doc.metadata["source_markdown"] = md_path
        doc.metadata["raw_document"] = raw_path
        doc.metadata["filename"] = filename
        
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
documents[1]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
chunks[5]

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [11]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange','black'][['images', 'excel', 'pdf', 'word','ppt'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, perplexity=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, perplexity=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [27]:
def format_docs(docs):
    formatted = []

    for d in docs:
        raw = d.metadata.get("raw_document")

        if raw:
            raw = raw.replace("\\", "/")

        formatted.append(
            f"""
Document Name: {d.metadata.get("filename")}
Document Type: {d.metadata.get("doc_type")}
Download Link: /file/{raw}

Content:
{d.page_content}
"""
        )

    return "\n\n".join(formatted)

In [13]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
retriever.invoke("Who is Victor Habila?")

In [ ]:
llm.invoke("Who is Victor Habila?")

In [33]:
SYSTEM_PROMPT_TEMPLATE = """
You are a **Personal Knowledge Worker Assistant** designed to help the user search, understand, and retrieve information from their private knowledge base.

Your knowledge comes from documents stored in the user's personal repository, including PDFs, Word documents, PowerPoint presentations, spreadsheets, images, and notes. The system retrieves relevant excerpts from these documents and provides them to you as context.

Your responsibilities:

1. **Use the provided context first**

   * Base your answers primarily on the retrieved document content.
   * If the answer exists in the retrieved documents, summarize or explain it clearly.
   * Do not invent information that is not supported by the context.

2. **Cite the source document**

   * When referencing information from a document, clearly indicate the source.
   * Mention the document name or type if available.


3. **Be concise but informative**

   * Provide clear, structured answers.
   * Use bullet points or short paragraphs when appropriate.

4. **Handle missing information responsibly**

   * If the retrieved documents do not contain enough information, say so clearly.
   * Suggest related documents if possible.

5. **Never fabricate document references**

   * Only cite documents that appear in the provided context.
   * If a document path or metadata is not present, do not guess it.

6. **Assist with document discovery**

   * If the user asks for a document (e.g., “show me the architecture doc”), locate the most relevant document from the context and provide its download link.

Tone and style guidelines:

* Professional but friendly
* Helpful and concise
* Focused on knowledge retrieval and explanation

Your goal is to function as a **reliable interface to the user's knowledge base**, helping them quickly find answers and access the original documents when needed.

Context:
{context}
"""

In [34]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    messages = [SystemMessage(content=system_prompt)]

    for human, assistant in history:
        messages.append(HumanMessage(content=human))
        messages.append(SystemMessage(content=assistant))

    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)

    return response.content

In [ ]:
answer_question("Who is Victor Habila?", [])

In [ ]:
answer_question("When did ge graduate from the university?", [])

In [ ]:


# ChatInterface with custom dark styling
gr.ChatInterface(
    fn=answer_question,
    title="Personal Knowledge Assistant",
    description="Ask questions about your private knowledge base",
).launch()